# 01 - Imagem -> Mesh (TripoSR) -> Point Cloud

Este notebook:
1. Clona o repositorio do projeto
2. Instala dependencias
3. Clona TripoSR
4. Roda inferencia nas imagens de `data/images/`
5. Salva meshes em `data/meshes/ground_truth/`
6. Converte meshes em point clouds (com normais) em `data/pointclouds/`

In [ ]:
# Cell 1 - Clonar repositorio e instalar dependencias
import os

REPO_DIR = "/content/cg_mesh"
if os.path.exists(REPO_DIR):
    !cd /content/cg_mesh && git pull
else:
    !git clone https://github.com/matheus2049alves/cg_mesh.git {REPO_DIR}

%cd {REPO_DIR}

!pip install -q -r requirements.txt
!pip install -q git+https://github.com/tatsy/torchmcubes.git

In [ ]:
# Cell 2 - Clonar TripoSR (pula se ja existe)
import os

TRIPOSR_DIR = "TripoSR"
if not os.path.exists(TRIPOSR_DIR):
    !git clone https://github.com/VAST-AI-Research/TripoSR.git
else:
    print(f"{TRIPOSR_DIR} ja existe, pulando clone.")

In [ ]:
# Cell 3 - Setup de paths
import os

PROJECT_ROOT = os.path.abspath(".")
DATA_IMAGES = os.path.join(PROJECT_ROOT, "data", "images")
DATA_MESHES_GT = os.path.join(PROJECT_ROOT, "data", "meshes", "ground_truth")
DATA_PC = os.path.join(PROJECT_ROOT, "data", "pointclouds")

os.makedirs(DATA_MESHES_GT, exist_ok=True)
os.makedirs(DATA_PC, exist_ok=True)

# Listar imagens disponiveis
image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
images = sorted([
    f for f in os.listdir(DATA_IMAGES)
    if os.path.splitext(f)[1].lower() in image_extensions
])
print(f"Imagens encontradas: {len(images)}")
for img in images:
    print(f"  - {img}")

In [ ]:
# Cell 4 - Carregar modelo TripoSR
import torch
from tsr.system import TSR

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(8192)
model.to(device)
print("TripoSR carregado com sucesso.")

In [ ]:
# Cell 5 - Processar imagens: imagem -> mesh
import numpy as np
from PIL import Image
import rembg
from tsr.utils import remove_background, resize_foreground

rembg_session = rembg.new_session()

mesh_paths = []

for img_name in images:
    img_path = os.path.join(DATA_IMAGES, img_name)
    mesh_name = os.path.splitext(img_name)[0] + ".obj"
    mesh_path = os.path.join(DATA_MESHES_GT, mesh_name)

    print(f"\nProcessando: {img_name}")

    # Load + remove bg
    image = Image.open(img_path)
    image = remove_background(image, rembg_session)
    image = resize_foreground(image, 0.85)
    image = np.array(image).astype(np.float32) / 255.0
    image = image[:, :, :3] * image[:, :, 3:4] + (1 - image[:, :, 3:4]) * 0.5
    image = Image.fromarray((image * 255.0).astype(np.uint8))

    # Inference
    with torch.no_grad():
        scene_codes = model([image], device=device)

    # Extract mesh
    meshes = model.extract_mesh(scene_codes, True, resolution=256)
    meshes[0].export(mesh_path)
    mesh_paths.append(mesh_path)
    print(f"  Mesh salva: {mesh_path}")

print(f"\nTotal de meshes geradas: {len(mesh_paths)}")

In [ ]:
# Cell 6 - Converter meshes -> point clouds com normais
import trimesh

SAMPLE_NUM = 8192

pc_paths = []

for mesh_path in mesh_paths:
    mesh_name = os.path.basename(mesh_path)
    pc_name = os.path.splitext(mesh_name)[0] + ".npy"
    pc_path = os.path.join(DATA_PC, pc_name)

    print(f"Convertendo: {mesh_name} -> {pc_name}")

    mesh = trimesh.load(mesh_path, force="mesh")

    # Amostrar pontos + normais das faces
    points, face_idx = mesh.sample(SAMPLE_NUM, return_index=True)
    normals = mesh.face_normals[face_idx]

    # Formato (N, 6): xyz + normals (conforme esperado pelo MeshAnythingV2)
    pc_normal = np.concatenate([points, normals], axis=-1, dtype=np.float16)
    np.save(pc_path, pc_normal)
    pc_paths.append(pc_path)
    print(f"  Point cloud salva: {pc_path} (shape: {pc_normal.shape})")

print(f"\nTotal de point clouds: {len(pc_paths)}")

In [ ]:
# Cell 7 - Resumo
print("=" * 60)
print("RESUMO - Passo 1: Imagem -> Mesh -> Point Cloud")
print("=" * 60)
print(f"\nImagens processadas: {len(images)}")
print(f"Meshes salvas em:    {DATA_MESHES_GT}")
print(f"Point clouds em:     {DATA_PC}")
print(f"\nProximo passo: 02_inference.ipynb (MeshAnything V2 + baselines)")